# Assignment 4: Model Development and Offline Evaluation

## Introduction

In this assignment I train, evaluate and compare machine learning models that predict the quality of red wine. The data is the Wine Quality dataset from the UCI Machine Learning Repository. Each row is one wine, described by its chemical properties, together with a quality score given by wine tasters.

I treat this as a multi-class classification problem. I group the quality scores into three classes, then follow two processes one after the other. First, I build two models by hand, a Logistic Regression and a Random Forest, and compare them using accuracy and F1-score. Second, I use an AutoML (Automated Machine Learning) library called TPOT to search for a model automatically, and I look at the log of the pipelines it tested. At the end, I compare the manual models with the AutoML result.

In [1]:
#Import libraries 
import pandas as pd

In [2]:
# I load the dataset
df = pd.read_csv("winequality-red.csv")

In [3]:
#I check the shape of the dataset
print("Shape of the data (rows, columns):", df.shape)


Shape of the data (rows, columns): (1599, 1)


In [4]:
# I look at the first 5 rows to see what the data looks like
df.head()


,"fixed acidity;""volatile acidity"";""citric acid"";""residual sugar"";""chlorides"";""free sulfur dioxide"";""total sulfur dioxide"";""density"";""pH"";""sulphates"";""alcohol"";""quality"""
0,7.4;0.7;0;1.9;0.076;11;34;0.9978;3.51;0.56;9.4;5
1,7.8;0.88;0;2.6;0.098;25;67;0.9968;3.2;0.68;9.8;5
2,7.8;0.76;0.04;2.3;0.092;15;54;0.997;3.26;0.65;...
3,11.2;0.28;0.56;1.9;0.075;17;60;0.998;3.16;0.58...
4,7.4;0.7;0;1.9;0.076;11;34;0.9978;3.51;0.56;9.4;5


In [5]:
df = pd.read_csv("winequality-red.csv", sep=";")

In [6]:
#I check the shape of the dataset
print("Shape of the data (rows, columns):", df.shape)

Shape of the data (rows, columns): (1599, 12)


In [7]:
# I look at the first 5 rows to see what the data looks like
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


**Interpretation**

I first loaded the file the normal way, without telling pandas what the separator is. The shape came back as 1,599 rows and only 1 column, and the first rows showed all the values squeezed together in a single column. This told me something was wrong with how the file was being read, not with the data itself.

Looking at that output, I could see a semicolon between every value. A CSV file normally uses commas, and pandas looks for commas by default, so it could not find where one column ends and the next begins. I loaded the file again with sep=";" and the shape changed to 1,599 rows and 12 columns.

Now each row is one red wine. Eleven columns describe its chemical properties, such as acidity, sugar, pH

In [8]:
#Data types and missing values
# I check the column names, how many values each column has, and the data type of each column
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [9]:
# I count the missing values in each column
print("\nMissing values in each column:")
print(df.isnull().sum())


Missing values in each column:
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64


In [10]:
# I add them up to get one total for the whole table
print("\nTotal missing values:", df.isnull().sum().sum())


Total missing values: 0


**Interpretation**

The table has 1,599 rows and 12 columns, and every column shows 1,599 non-null values. This means no value is missing anywhere, which the missing value count confirms with a total of 0. The assignment asks me to handle missing values if they are present, and here there are none, so I do not need to drop or fill anything.

Eleven columns are float64, which means numbers with decimals. These are the chemical measurements of the wine and they will be my features. The last column, quality, is int64, a whole number, and it is the target I want to predict. There are no text columns, so I do not need any encoding before training a model.

In [11]:
# I count how many rows are an exact copy of an earlier row
duplicate_count = df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 240


In [12]:
# I work out what share of the data the duplicates are
duplicate_percent = duplicate_count / len(df) * 100
print("Share of the data:", round(duplicate_percent, 1), "%")

Share of the data: 15.0 %


In [13]:
# I look at a few of the repeated rows, with the copies placed next to each other
df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(6)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
142,5.2,0.34,0.00,1.8,0.050,27.0,63.0,0.9916,3.68,0.79,14.0,6
144,5.2,0.34,0.00,1.8,0.050,27.0,63.0,0.9916,3.68,0.79,14.0,6
131,5.6,0.50,0.09,2.3,0.049,17.0,99.0,0.9937,3.63,0.63,13.0,5
132,5.6,0.50,0.09,2.3,0.049,17.0,99.0,0.9937,3.63,0.63,13.0,5
1488,5.6,0.54,0.04,1.7,0.049,5.0,13.0,0.9942,3.72,0.58,11.4,5
1491,5.6,0.54,0.04,1.7,0.049,5.0,13.0,0.9942,3.72,0.58,11.4,5


**Interpretation**

I found 240 duplicate rows, which is 15% of the data. When I placed the copies next to each other, I could see that they match on every chemical measurement and also on the quality score, so they are exact repeats and not just similar wines.

I decided to remove them. Later I will split the data into a training set and a test set. If one copy of a wine goes into training and the other copy goes into testing, the model is tested on a row it has already seen. This is a form of data leakage, and it would make my accuracy and F1-score look better than they really are. Removing the copies gives me less data, but a fairer test.

In [14]:
# Drop the duplicates
# I remove the duplicate rows and keep the first appearance of each wine
df = df.drop_duplicates()

In [15]:
# I reset the row numbers so they run 0, 1, 2, ... with no gaps
df = df.reset_index(drop=True)

In [16]:
# I check the new shape and confirm no duplicates are left
print("Shape after removing duplicates:", df.shape)
print("Duplicates left:", df.duplicated().sum())

Shape after removing duplicates: (1359, 12)
Duplicates left: 0


**Interpretation**

After removing the duplicates, the table went from 1,599 rows to 1,359 rows, and the number of columns stayed at 12. The second check shows 0 duplicates left, so every row is now a different wine. I also reset the row numbers so they run from 0 to 1,358 with no gaps. This cleaned table is what I use for the rest of the assignment.

# Part 2: Map the Quality Scores into Classes

In [17]:
# I count how many wines have each quality score, ordered from the lowest score to the highest
quality_counts = df["quality"].value_counts().sort_index()
print("Number of wines at each quality score:")
print(quality_counts)

Number of wines at each quality score:
quality
3     10
4     53
5    577
6    535
7    167
8     17
Name: count, dtype: int64


In [18]:
# I show the same thing as a percentage of all wines
quality_percent = df["quality"].value_counts(normalize=True).sort_index() * 100
print("\nPercentage of wines at each quality score:")
print(quality_percent.round(1))


Percentage of wines at each quality score:
quality
3     0.7
4     3.9
5    42.5
6    39.4
7    12.3
8     1.3
Name: proportion, dtype: float64


**Interpretation**

The quality scale goes from 0 to 10, but in this data only the scores 3 to 8 appear. Most wines sit in the middle: score 5 has 577 wines (42.5%) and score 6 has 535 wines (39.4%), so these two scores together make up about 82% of the data. The ends are very rare. Only 10 wines have a score of 3 and only 17 wines have a score of 8.

This is a class imbalance. If I kept all six scores as separate classes, a model would have almost no examples to learn the rarest scores from. For this reason I group the scores into three classes in the next step, which gives each class more examples and makes the problem simpler.

In [19]:
#Map the scores into three classes


In [20]:
# I write a small rule that turns a quality score into a class number
# 0 = Low (score 5 or below), 1 = Medium (score 6), 2 = High (score 7 or above)
def map_quality(score):
    if score <= 5:
        return 0
    elif score == 6:
        return 1
    else:
        return 2

In [21]:
# I apply the rule to every wine and save the result in a new column
df["quality_class"] = df["quality"].apply(map_quality)

In [22]:
# I keep the class names in one place so I can use them in my tables and reports later
class_names = {0: "Low", 1: "Medium", 2: "High"}

In [23]:
# I count the wines in each class and work out the percentage
class_counts = df["quality_class"].value_counts().sort_index()
class_percent = (class_counts / len(df) * 100).round(1)

In [24]:
# I put the name, the count and the percentage into one small table
class_summary = pd.DataFrame({
    "Class name": class_counts.index.map(class_names),
    "Wines": class_counts,
    "Percent": class_percent
})
print(class_summary)

              Class name  Wines  Percent
quality_class                           
0                    Low    640     47.1
1                 Medium    535     39.4
2                   High    184     13.5


In [25]:
# I check that every original score went into the class I intended
print("\nOriginal score (rows) against new class (columns):")
print(pd.crosstab(df["quality"], df["quality_class"]))


Original score (rows) against new class (columns):
quality_class    0    1    2
quality                     
3               10    0    0
4               53    0    0
5              577    0    0
6                0  535    0
7                0    0  167
8                0    0   17


**Interpretation**

I grouped the six quality scores into three classes. Wines with a score of 5 or below became Low (class 0), wines with a score of 6 became Medium (class 1), and wines with a score of 7 or above became High (class 2). I stored the classes as numbers because models work with number labels, and I kept the names in a small dictionary so my results are easier to read.

The Low class has 640 wines (47.1%), the Medium class has 535 wines (39.4%) and the High class has 184 wines (13.5%). This is much more balanced than the original scores, where the rarest score had only 10 wines. The High class is still the smallest, so I need to remember this when I split the data and when I judge the models.

The cross table confirms that the rule worked as I intended. Scores 3, 4 and 5 appear only under class 0, score 6 appears only under class 1, and scores 7 and 8 appear only under class 2. No wine was placed in the wrong class.

# Part 3: Split the Data and Standardize

In [26]:
#Features, target and the stratified split
from sklearn.model_selection import train_test_split

In [27]:
# X holds my features: the 11 chemical measurements
# I remove both quality columns, because the class was made from the quality score
X = df.drop(columns=["quality", "quality_class"])

In [28]:
# y holds my target: the class I want to predict
y = df["quality_class"]

In [29]:
# I split the data: 80% for training and 20% for testing
# stratify=y keeps the same share of each class in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [30]:
# I check the size of each set
print("Training set:", X_train.shape)
print("Test set:    ", X_test.shape)

Training set: (1087, 11)
Test set:     (272, 11)


In [31]:
# I compare the class percentages in the full data, the training set and the test set
split_check = pd.DataFrame({
    "Full data %": y.value_counts(normalize=True).sort_index() * 100,
    "Train %": y_train.value_counts(normalize=True).sort_index() * 100,
    "Test %": y_test.value_counts(normalize=True).sort_index() * 100,
    "Train wines": y_train.value_counts().sort_index(),
    "Test wines": y_test.value_counts().sort_index()
}).round(1)
print("\nClass share in each set:")
print(split_check)


Class share in each set:
               Full data %  Train %  Test %  Train wines  Test wines
quality_class                                                       
0                     47.1     47.1    47.1          512         128
1                     39.4     39.4    39.3          428         107
2                     13.5     13.5    13.6          147          37


**Interpretation**

I separated the data into features and a target. The features (X) are the 11 chemical measurements. The target (y) is the quality class. I removed the original quality score from the features, because the class was made directly from that score. If the model could see the score, it would simply read the answer, which is target leakage.

I then split the data into 80% for training and 20% for testing. The training set has 1,087 wines and the test set has 272 wines, and both have 11 feature columns. The model learns from the training set, and the test set stays hidden until the end so I can check how the model performs on wines it has never seen.

I used a stratified split, and the table shows that it worked. The Low class is 47.1% in the full data, in the training set and in the test set. The Medium class is 39.4% in training and 39.3% in testing, and the High class is 13.5% in training and 13.6% in testing. The tiny differences come from rounding to whole wines. The High class is the smallest, with 147 wines for training and only 37 for testing, so I will look at how the models do on this class in particular.

In [32]:
#Standardize with StandardScaler
from sklearn.preprocessing import StandardScaler

In [33]:
# I look at the features before scaling, to see how different their ranges are
print("Training features BEFORE scaling:")
print(X_train.describe().loc[["mean", "std", "min", "max"]].T.round(2))

Training features BEFORE scaling:
                       mean    std   min     max
fixed acidity          8.32   1.78  4.70   15.90
volatile acidity       0.53   0.18  0.12    1.58
citric acid            0.27   0.20  0.00    1.00
residual sugar         2.53   1.36  0.90   15.50
chlorides              0.09   0.05  0.01    0.61
free sulfur dioxide   15.85  10.50  1.00   72.00
total sulfur dioxide  47.34  34.33  6.00  289.00
density                1.00   0.00  0.99    1.00
pH                     3.31   0.16  2.74    4.01
sulphates              0.66   0.17  0.37    2.00
alcohol               10.44   1.09  8.40   14.90


In [34]:
# I create the scaler
scaler = StandardScaler()

# I fit the scaler on the training set only, and transform the training set
# fit = learn the mean and standard deviation of each column
# transform = use them to rescale the values
X_train_scaled = scaler.fit_transform(X_train)

# I transform the test set with the SAME mean and standard deviation learned from training
# I do not fit again, so nothing from the test set leaks into my preparation
X_test_scaled = scaler.transform(X_test)

In [35]:
# I check the result: mean and standard deviation of each feature after scaling
scale_check = pd.DataFrame({
    "Train mean": X_train_scaled.mean(axis=0),
    "Train std": X_train_scaled.std(axis=0),
    "Test mean": X_test_scaled.mean(axis=0),
    "Test std": X_test_scaled.std(axis=0)
}, index=X_train.columns).round(2)
print("\nAFTER scaling:")
print(scale_check)


AFTER scaling:
                      Train mean  Train std  Test mean  Test std
fixed acidity               -0.0        1.0      -0.02      0.88
volatile acidity            -0.0        1.0       0.01      0.98
citric acid                  0.0        1.0      -0.05      0.95
residual sugar              -0.0        1.0      -0.02      0.98
chlorides                   -0.0        1.0       0.02      0.98
free sulfur dioxide         -0.0        1.0       0.02      0.98
total sulfur dioxide        -0.0        1.0      -0.08      0.85
density                      0.0        1.0      -0.00      0.90
pH                           0.0        1.0      -0.02      0.98
sulphates                    0.0        1.0       0.01      1.00
alcohol                     -0.0        1.0      -0.03      0.94


**Interpretation**

Before scaling, the 11 features were measured on very different scales. Some features have large values and others have very small ones. This matters for a model like Logistic Regression, because a feature with large numbers can dominate the model just because of its size and not because it is more important.

I used StandardScaler, which changes each feature so that its mean becomes 0 and its standard deviation becomes 1. I fitted the scaler on the training set only and then used the same scaler to transform the test set. This way, nothing from the test set is used while preparing the data, which avoids data leakage.

The table confirms this. In the training set, every feature has a mean of 0 and a standard deviation of 1. Some means show as -0.0, which is just a tiny rounding effect and is the same as 0. In the test set, the means are close to 0 (between -0.08 and 0.02) and the standard deviations are close to 1 (between 0.85 and 1.00), but they are not exact. This is expected and correct, because the test set was scaled with the mean and standard deviation of the training set and not its own. All features are now on the same scale and ready for modelling.

# Part 4: Train and Evaluate a Logistic Regression Model

In [36]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [37]:
# I create the Logistic Regression model
lr_model = LogisticRegression(max_iter=1000, random_state=42)

In [38]:
# I train the model on the scaled training data
lr_model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [39]:
# I ask the model to predict the class of every wine in the test set
lr_pred = lr_model.predict(X_test_scaled)

In [40]:
# I measure the model on the test set and save the scores for my comparison later
lr_accuracy = accuracy_score(y_test, lr_pred)
lr_f1_macro = f1_score(y_test, lr_pred, average="macro")
lr_f1_weighted = f1_score(y_test, lr_pred, average="weighted")

In [41]:
print("Logistic Regression results on the test set")
print("Accuracy:           ", round(lr_accuracy, 3))
print("F1-score (macro):   ", round(lr_f1_macro, 3))
print("F1-score (weighted):", round(lr_f1_weighted, 3))

# I print the detailed report, which shows precision, recall and F1 for each class
print("\nClassification report:")
print(classification_report(y_test, lr_pred, target_names=list(class_names.values())))

Logistic Regression results on the test set
Accuracy:            0.61
F1-score (macro):    0.547
F1-score (weighted): 0.6

Classification report:
              precision    recall  f1-score   support

         Low       0.69      0.77      0.72       128
      Medium       0.52      0.53      0.53       107
        High       0.58      0.30      0.39        37

    accuracy                           0.61       272
   macro avg       0.59      0.53      0.55       272
weighted avg       0.61      0.61      0.60       272



**Interpretation**

The Logistic Regression model predicted the correct class for 61% of the test wines. For comparison, a model that always guessed the largest class, Low, would be right about 47% of the time, so the model has learned something real, but it is far from perfect.

The results are very different from class to class. The Low class is the strongest. Its recall is 0.77, which means the model found 77% of the wines that really are Low, and its precision is 0.69, which means that when the model says a wine is Low, it is right 69% of the time. The Medium class is weaker, with a precision of 0.52 and a recall of 0.53. The model finds only about half of the Medium wines and is right only about half of the time when it predicts Medium. This makes sense, because Medium sits between the other two classes and is easy to confuse with both.

The High class is the weakest. Its recall is only 0.30, so out of the 37 wines that really are High, the model found only about 11 and missed the rest. Its precision is 0.58, so when the model does predict High, it is right a little more than half of the time. This is the smallest class, and the model had fewer examples to learn it from.

The two F1-scores show the same story. The weighted F1-score is 0.60, which is close to the accuracy because the larger classes count more. The macro F1-score is lower at 0.547, because it treats all three classes equally and the weak High class pulls it down. This gap tells me that accuracy alone hides the problem with the High class.

# Part 5: Train and Evaluate a Random Forest Model


In [42]:
from sklearn.ensemble import RandomForestClassifier

In [43]:
# I create the Random Forest model with 100 decision trees
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

In [44]:
# I train the model on the same scaled training data as before, so the comparison is fair
rf_model.fit(X_train_scaled, y_train)


RandomForestClassifier(random_state=42)

In [45]:

# I ask the model to predict the class of every wine in the test set
rf_pred = rf_model.predict(X_test_scaled)

In [46]:
# I measure the model on the test set and save the scores for my comparison later
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_f1_macro = f1_score(y_test, rf_pred, average="macro")
rf_f1_weighted = f1_score(y_test, rf_pred, average="weighted")

In [47]:
print("Random Forest results on the test set")
print("Accuracy:           ", round(rf_accuracy, 3))
print("F1-score (macro):   ", round(rf_f1_macro, 3))
print("F1-score (weighted):", round(rf_f1_weighted, 3))

# I print the detailed report, which shows precision, recall and F1 for each class
print("\nClassification report:")
print(classification_report(y_test, rf_pred, target_names=list(class_names.values())))

Random Forest results on the test set
Accuracy:            0.632
F1-score (macro):    0.566
F1-score (weighted): 0.622

Classification report:
              precision    recall  f1-score   support

         Low       0.70      0.80      0.75       128
      Medium       0.56      0.54      0.55       107
        High       0.52      0.32      0.40        37

    accuracy                           0.63       272
   macro avg       0.59      0.55      0.57       272
weighted avg       0.62      0.63      0.62       272



**Interpretation**

The Random Forest model predicted the correct class for 63.2% of the test wines. This is slightly better than Logistic Regression, which reached 61%. The macro F1-score is 0.566, compared with 0.547 for Logistic Regression, and the weighted F1-score is 0.622, compared with 0.600. So Random Forest is a little higher on all three measures.

The Low class is again the strongest. When the model predicts Low, it is right 70% of the time (precision 0.70). Out of all the wines that really are Low, it found 80% and missed 20% (recall 0.80). The Medium class improved a little, with a precision of 0.56 and a recall of 0.54, but the model still finds only about half of the Medium wines.

The High class is still the weak point. Its recall is 0.32, so out of the 37 wines that really are High, the model found only about 12. Its precision is 0.52, which means that when the model predicts High, it is right about half of the time. This is almost the same as Logistic Regression, so changing the model did not solve the problem with the smallest class.

Overall, Random Forest is the better of the two models, but the improvement is small, about 2 percentage points in accuracy and in both F1-scores. In both models the macro F1-score is lower than the weighted F1-score, because the macro score treats all three classes equally and the weak High class pulls it down. Both models show the same pattern: good on Low, average on Medium and poor on High.

In [48]:
# I put the saved scores of both models side by side in one table
comparison = pd.DataFrame({
    "Logistic Regression": [lr_accuracy, lr_f1_macro, lr_f1_weighted],
    "Random Forest": [rf_accuracy, rf_f1_macro, rf_f1_weighted]
}, index=["Accuracy", "F1-score (macro)", "F1-score (weighted)"])

# I add a column that shows how much higher Random Forest is
comparison["Difference"] = comparison["Random Forest"] - comparison["Logistic Regression"]
print(comparison.round(3))

# I count how many of the 272 test wines each model predicted correctly
print("\nWines predicted correctly out of", len(y_test))
print("Logistic Regression:", (lr_pred == y_test).sum())
print("Random Forest:      ", (rf_pred == y_test).sum())

                     Logistic Regression  Random Forest  Difference
Accuracy                           0.610          0.632       0.022
F1-score (macro)                   0.547          0.566       0.019
F1-score (weighted)                0.600          0.622       0.022

Wines predicted correctly out of 272
Logistic Regression: 166
Random Forest:       172


**Comparison and Reflection**

Random Forest performed better than Logistic Regression on all three measures. Its accuracy is 0.632 against 0.610, its macro F1-score is 0.566 against 0.547, and its weighted F1-score is 0.622 against 0.600. In real numbers, Random Forest predicted 172 of the 272 test wines correctly and Logistic Regression predicted 166, so the difference is 6 wines.

I think Random Forest did better because of the kind of model it is. Logistic Regression is a linear model. It gives each feature one weight and adds them up, so it can only learn straight-line relationships. Wine quality probably does not work this way. A feature like acidity may be good up to a point and then bad, and features may matter in combination, such as alcohol together with sulphates. Random Forest is built from 100 decision trees that vote, and trees can learn these non-linear patterns and combinations by themselves. Because many trees vote together, the mistakes of one tree are also balanced out by the others.

At the same time, the improvement is small, and both models are far from perfect. I think the main reason is the data and not the models. The three classes overlap: a wine with a score of 5 and a wine with a score of 6 can have very similar chemistry, and the scores come from human tasters, so they are partly a matter of opinion. The dataset is also small, with 1,087 wines for training, and the High class has only 147 of them. Both models found fewer than one third of the High wines, so the smallest class is the weak point for both.

There is also a trade-off. Logistic Regression is simpler, faster and easier to explain, because each feature has one weight that I can read. Random Forest is more complex and harder to explain, but it is a little more accurate. With a difference of only 6 wines on this test set, I would say Random Forest is the better model here, but not by a large margin.

### Improving the High Class with Class Weights

In [49]:
from sklearn.metrics import recall_score

In [50]:
# I write a small helper that collects the scores I care about for one set of predictions
def score_row(model_name, predictions):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "F1 (macro)": f1_score(y_test, predictions, average="macro"),
        "High recall": recall_score(y_test, predictions, average=None)[2]
    }

In [51]:
# I create the same two models again, this time with balanced class weights
# class_weight="balanced" makes a mistake on a small class cost more than a mistake on a big class
lr_balanced = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
rf_balanced = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")


In [52]:
# I train both models on the same training data and predict the test set
lr_balanced.fit(X_train_scaled, y_train)
rf_balanced.fit(X_train_scaled, y_train)
lr_balanced_pred = lr_balanced.predict(X_test_scaled)
rf_balanced_pred = rf_balanced.predict(X_test_scaled)

In [53]:
# I put the original models and the balanced models into one table
weights_table = pd.DataFrame([
    score_row("Logistic Regression (original)", lr_pred),
    score_row("Logistic Regression (balanced)", lr_balanced_pred),
    score_row("Random Forest (original)", rf_pred),
    score_row("Random Forest (balanced)", rf_balanced_pred)
]).set_index("Model")
print(weights_table.round(3))

                                Accuracy  F1 (macro)  High recall
Model                                                            
Logistic Regression (original)     0.610       0.547        0.297
Logistic Regression (balanced)     0.596       0.574        0.730
Random Forest (original)           0.632       0.566        0.324
Random Forest (balanced)           0.621       0.565        0.378


**Interpretation**

I trained both models again with class_weight="balanced", which makes a mistake on a small class cost more during training. The data, the split and the test set stayed exactly the same.

Logistic Regression changed a lot. Its recall for the High class went from 0.297 to 0.730, so instead of finding about 11 of the 37 High wines, it now finds about 27. Its macro F1-score also went up, from 0.547 to 0.574, which is the highest macro F1-score of the four models. The price is a small drop in accuracy, from 0.610 to 0.596. The model is now more willing to predict High, so it catches many more High wines, but it also gets a few more wines from the other classes wrong.

Random Forest hardly changed. Its High recall went from 0.324 to 0.378, which is only about 2 more wines, its macro F1-score stayed almost the same (0.566 and 0.565) and its accuracy dropped slightly from 0.632 to 0.621. The class weights have much less effect on this model. A Logistic Regression has one decision line that the weights can push directly, while a Random Forest is made

# Part 7: Automated Model Selection with TPOT

In [54]:
from tpot import TPOTClassifier

C:\Users\patel\tpot-env\Lib\site-packages\stopit\__init__.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
C:\Users\patel\tpot-env\Lib\site-packages\tpot\builtins\__init__.py:36: UserWarning: Warning: optional dependency `torch` is not available. - skipping import of NN models.
  warnings.warn("Warning: optional dependency `torch` is not available. - skipping import of NN models.")


In [55]:
# I hide the long library warnings so the search log is easy to read
import warnings
warnings.filterwarnings("ignore")

In [56]:
# I set up the search
tpot = TPOTClassifier(
    generations=5,          # how many rounds of improvement TPOT runs
    population_size=20,     # how many pipelines it keeps in each round
    cv=5,                   # each pipeline is tested with 5-fold cross-validation
    scoring="accuracy",     # the score TPOT tries to make as high as possible
    random_state=42,        # keeps the search the same every time I run it
    verbosity=2             # prints a progress bar and the best score after each generation
)


In [57]:
# I run the search on the training data only
# The test set stays hidden, exactly as it did for my manual models
tpot.fit(X_train_scaled, y_train)

Version 0.12.2 of tpot is outdated. Version 1.1.0 was released Thursday July 03, 2025.


                                                                             
Generation 1 - Current best internal CV score: 0.6338730816386928
                                                                             
Generation 2 - Current best internal CV score: 0.6338730816386928
                                                                             
Generation 3 - Current best internal CV score: 0.6357037162304993
                                                                              
Generation 4 - Current best internal CV score: 0.6357037162304993
                                                                              
Generation 5 - Current best internal CV score: 0.6366211474231598
                                                           
Best pipeline: RandomForestClassifier(PolynomialFeatures(input_matrix, degree=2, include_bias=False, interaction_only=False), bootstrap=False, criterion=gini, max_features=0.2, min_samples_leaf=6, min_samples_split=4, 

TPOTClassifier(generations=5, population_size=20, random_state=42,
               scoring='accuracy', verbosity=2)

In [58]:
# I show the best pipeline TPOT found
print("\nBest pipeline found by TPOT:")
print(tpot.fitted_pipeline_)



Best pipeline found by TPOT:
Pipeline(steps=[('polynomialfeatures', PolynomialFeatures(include_bias=False)),
                ('randomforestclassifier',
                 RandomForestClassifier(bootstrap=False, max_features=0.2,
                                        min_samples_leaf=6, min_samples_split=4,
                                        random_state=42))])


In [ ]:
Generation 1 - Current best internal CV score: 0.6...
...
Best pipeline: RandomForestClassifier(PolynomialFeatures(...

**Interpretation**

I used TPOT to search for a model automatically. I set it to run 5 generations with 20 pipelines in each, which gives up to 120 pipelines in total. TPOT only used the training set. It judged every pipeline with 5-fold cross-validation, which means it split the training set into 5 parts, trained on 4 parts and tested on the remaining part, five times, and took the average accuracy. The test set stayed hidden the whole time.

The log shows how the best score changed from one generation to the next. After generation 1 the best cross-validation accuracy was 0.6339. It stayed the same in generation 2, rose to 0.6357 in generation 3, stayed the same in generation 4 and ended at 0.6366 in generation 5. The total improvement over the whole search is very small, less than 0.003. This tells me that TPOT found a good pipeline very early and that the later generations could only improve it a little.

The best pipeline has two steps. The first step, PolynomialFeatures, creates new features by squaring each feature and multiplying every pair of features together, which turns my 11 features into 77. This is feature crossing done automatically. The second step is a Random Forest with tuned settings. Each tree uses all the training rows, looks at only 20% of the features at each split, and must keep at least 6 wines in every final group, which helps to stop the trees from memorising single wines.

It is interesting that TPOT chose a Random Forest as its final model, which is the same type of model that won my manual comparison. The difference is that TPOT added a feature creation step and searched for the settings, while I used the default settings.

# Part 8: Log of All Pipelines Tested by TPOT

In [59]:
# TPOT keeps a record of every pipeline it tested; I turn that record into a table
tpot_log = pd.DataFrame.from_dict(tpot.evaluated_individuals_, orient="index")

# The pipeline description is in the row labels, so I move it into a normal column
tpot_log = tpot_log.rename_axis("Pipeline").reset_index()

# I pull out the name of the final model in each pipeline (the first word before the bracket)
tpot_log["Model"] = tpot_log["Pipeline"].str.split("(").str[0]

# I keep the columns I need and sort from the best score to the worst
tpot_log = tpot_log[["Model", "generation", "operator_count", "internal_cv_score", "Pipeline"]]
tpot_log = tpot_log.sort_values("internal_cv_score", ascending=False).reset_index(drop=True)

# I tell pandas to show every row and the full pipeline text without cutting it off
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

print("Total pipelines tested:", len(tpot_log))

# I save the full log to a file, so I have a record outside the notebook
tpot_log.to_csv("tpot_pipeline_log.csv", index=False)

# I show the full log
tpot_log

Total pipelines tested: 116


,Model,generation,operator_count,internal_cv_score,Pipeline
0,RandomForestClassifier,5,2.0,0.636621,"RandomForestClassifier(PolynomialFeatures(input_matrix, PolynomialFeatures__degree=2, PolynomialFeatures__include_bias=False, PolynomialFeatures__interaction_only=False), RandomForestClassifier__bootstrap=False, RandomForestClassifier__criterion=gini, RandomForestClassifier__max_features=0.2, RandomForestClassifier__min_samples_leaf=6, RandomForestClassifier__min_samples_split=4, RandomForestClassifier__n_estimators=100)"
1,RandomForestClassifier,3,1.0,0.635704,"RandomForestClassifier(input_matrix, RandomForestClassifier__bootstrap=False, RandomForestClassifier__criterion=gini, RandomForestClassifier__max_features=0.25, RandomForestClassifier__min_samples_leaf=2, RandomForestClassifier__min_samples_split=12, RandomForestClassifier__n_estimators=100)"
2,RandomForestClassifier,4,2.0,0.635704,"RandomForestClassifier(OneHotEncoder(input_matrix, OneHotEncoder__minimum_fraction=0.1, OneHotEncoder__sparse=False, OneHotEncoder__threshold=10), RandomForestClassifier__bootstrap=False, RandomForestClassifier__criterion=gini, RandomForestClassifier__max_features=0.25, RandomForestClassifier__min_samples_leaf=2, RandomForestClassifier__min_samples_split=12, RandomForestClassifier__n_estimators=100)"
3,RandomForestClassifier,4,3.0,0.635699,"RandomForestClassifier(MinMaxScaler(PolynomialFeatures(input_matrix, PolynomialFeatures__degree=2, PolynomialFeatures__include_bias=False, PolynomialFeatures__interaction_only=False)), RandomForestClassifier__bootstrap=False, RandomForestClassifier__criterion=gini, RandomForestClassifier__max_features=0.2, RandomForestClassifier__min_samples_leaf=6, RandomForestClassifier__min_samples_split=4, RandomForestClassifier__n_estimators=100)"
4,RandomForestClassifier,5,2.0,0.634778,"RandomForestClassifier(PolynomialFeatures(input_matrix, PolynomialFeatures__degree=2, PolynomialFeatures__include_bias=False, PolynomialFeatures__interaction_only=False), RandomForestClassifier__bootstrap=True, RandomForestClassifier__criterion=entropy, RandomForestClassifier__max_features=0.2, RandomForestClassifier__min_samples_leaf=8, RandomForestClassifier__min_samples_split=4, RandomForestClassifier__n_estimators=100)"
5,RandomForestClassifier,0,2.0,0.633873,"RandomForestClassifier(PolynomialFeatures(input_matrix, PolynomialFeatures__degree=2, PolynomialFeatures__include_bias=False, PolynomialFeatures__interaction_only=False), RandomForestClassifier__bootstrap=False, RandomForestClassifier__criterion=gini, RandomForestClassifier__max_features=0.25, RandomForestClassifier__min_samples_leaf=2, RandomForestClassifier__min_samples_split=8, RandomForestClassifier__n_estimators=100)"
6,RandomForestClassifier,2,1.0,0.633848,"RandomForestClassifier(input_matrix, RandomForestClassifier__bootstrap=False, RandomForestClassifier__criterion=gini, RandomForestClassifier__max_features=0.25, RandomForestClassifier__min_samples_leaf=2, RandomForestClassifier__min_samples_split=14, RandomForestClassifier__n_estimators=100)"
7,RandomForestClassifier,3,2.0,0.632960,"RandomForestClassifier(PolynomialFeatures(input_matrix, PolynomialFeatures__degree=2, PolynomialFeatures__include_bias=False, PolynomialFeatures__interaction_only=False), RandomForestClassifier__bootstrap=True, RandomForestClassifier__criterion=gini, RandomForestClassifier__max_features=0.2, RandomForestClassifier__min_samples_leaf=8, RandomForestClassifier__min_samples_split=4, RandomForestClassifier__n_estimators=100)"
8,ExtraTreesClassifier,5,1.0,0.632918,"ExtraTreesClassifier(input_matrix, ExtraTreesClassifier__bootstrap=False, ExtraTreesClassifier__criterion=gini, ExtraTreesClassifier__max_features=0.8, ExtraTreesClassifier__min_samples_leaf=18, ExtraTreesClassifier__min_samples_split=5, ExtraTreesClassifier__n_estimators=100)"
9,RandomForestClassifier,5,1.0,0.632013,"RandomForestClassifier(input_matrix, RandomForestClassifier__bootstrap=True, RandomForestClassifier__criterion=gini

In [60]:
# I leave out the pipelines that failed (their score is minus infinity)
valid_log = tpot_log[tpot_log["internal_cv_score"] > 0]
print("Pipelines that ran successfully:", len(valid_log))
print("Pipelines that failed:          ", len(tpot_log) - len(valid_log))

# For each type of final model, I count the pipelines and find the best, average and worst score
model_summary = valid_log.groupby("Model")["internal_cv_score"].agg(["count", "max", "mean", "min"])
model_summary = model_summary.sort_values("max", ascending=False).round(3)
print("\nSummary by final model:")
print(model_summary)

Pipelines that ran successfully: 113
Pipelines that failed:           3

Summary by final model:
                            count    max   mean    min
Model                                                 
RandomForestClassifier         32  0.637  0.629  0.611
ExtraTreesClassifier           41  0.633  0.616  0.547
LogisticRegression              2  0.626  0.566  0.506
MLPClassifier                   2  0.622  0.595  0.568
XGBClassifier                   7  0.620  0.586  0.471
GaussianNB                      9  0.618  0.589  0.443
KNeighborsClassifier            8  0.616  0.589  0.546
GradientBoostingClassifier      3  0.607  0.556  0.471
DecisionTreeClassifier          3  0.593  0.574  0.555
LinearSVC                       1  0.590  0.590  0.590
BernoulliNB                     3  0.567  0.559  0.545
SGDClassifier                   2  0.422  0.416  0.409


**Interpretation**

TPOT keeps a record of every pipeline it tested, and I turned this record into a table and saved it as a CSV file. The setting allowed up to 120 pipelines, but the log has 116. This is because TPOT creates new pipelines by changing or combining earlier ones, and sometimes the result is an exact copy of a pipeline it has already scored. It does not test the same pipeline twice. Of the 116 pipelines, 113 ran successfully and 3 failed. The 3 that failed had created thousands of features, for example by applying PolynomialFeatures twice, so they were too large to run.

TPOT tested 12 different types of final model, but it did not spend its time equally. Random Forest was the final model in 32 of the successful pipelines and Extra Trees, which is a very similar model, in 41. Together they make up about 65% of the search. The reason is the way TPOT works. The first generation is random and has a lot of variety. After that, TPOT keeps the pipelines with the best scores and builds the next generation from them by making small changes. Because the tree models scored best, the later generations are mostly the same model with different settings or with a different preparation step in front of it.

Random Forest was the strongest and also the most stable model. Its best score was 0.637, its average was 0.629 and even its worst pipeline scored 0.611. Extra Trees came second with a best score of 0.633. Logistic Regression was tried only twice, with a best score of 0.626, and the weakest model was the SGD classifier, with a best score of only 0.422. Some pipelines scored 0.471, which is the share of Low wines in the training set, so those models simply predicted Low for every wine and learned nothing.

The log also shows that the best scores are very close to each other. The winning pipeline scored 0.6366 with the extra PolynomialFeatures step, and a plain Random Forest with no extra step scored 0.6357. This tells me the automatic search found the right type of model, but the fine-tuning added very little.

# Part 9: AutoML Compared with the Manual Models

In [62]:
# I ask TPOT's best pipeline to predict the class of every wine in the test set
# This is the first time the pipeline sees the test set
tpot_pred = tpot.predict(X_test_scaled)

# I print the detailed report for the TPOT pipeline
print("TPOT best pipeline: classification report on the test set")
print(classification_report(y_test, tpot_pred, target_names=list(class_names.values())))

TPOT best pipeline: classification report on the test set
              precision    recall  f1-score   support

         Low       0.72      0.81      0.76       128
      Medium       0.57      0.54      0.56       107
        High       0.54      0.38      0.44        37

    accuracy                           0.65       272
   macro avg       0.61      0.58      0.59       272
weighted avg       0.64      0.65      0.64       272



In [63]:
# I compare the score TPOT expected (cross-validation) with the score it got on the test set
print("Cross-validation accuracy (training set):", round(tpot_log.loc[0, "internal_cv_score"], 3))
print("Test accuracy:                           ", round(accuracy_score(y_test, tpot_pred), 3))



Cross-validation accuracy (training set): 0.637
Test accuracy:                            0.647


In [67]:
# I put my manual models and the TPOT pipeline into one final table
final_table = pd.DataFrame([
    score_row("Logistic Regression (manual)", lr_pred),
    score_row("Random Forest (manual)", rf_pred),
    score_row("Logistic Regression balanced (manual)", lr_balanced_pred),
    score_row("TPOT best pipeline (AutoML)", tpot_pred)
]).set_index("Model")

# I add the number of test wines each model predicted correctly
final_table["Correct of 272"] = [
    (lr_pred == y_test).sum(),
    (rf_pred == y_test).sum(),
    (lr_balanced_pred == y_test).sum(),
    (tpot_pred == y_test).sum()
]
print("\nFinal comparison on the test set:")
print(final_table.round(3))


Final comparison on the test set:
                                       Accuracy  F1 (macro)  High recall  \
Model                                                                      
Logistic Regression (manual)              0.610       0.547        0.297   
Random Forest (manual)                    0.632       0.566        0.324   
Logistic Regression balanced (manual)     0.596       0.574        0.730   
TPOT best pipeline (AutoML)               0.647       0.588        0.378   

                                       Correct of 272  
Model                                                  
Logistic Regression (manual)                      166  
Random Forest (manual)                            172  
Logistic Regression balanced (manual)             162  
TPOT best pipeline (AutoML)                       176  


**Interpretation**

I tested the best TPOT pipeline on the test set, which it had never seen. It predicted 176 of the 272 wines correctly, an accuracy of 0.647. Its macro F1-score is 0.588. Both are the highest of all the models in this assignment. Compared with my manual Random Forest, TPOT is 4 wines better (176 against 172), and compared with my Logistic Regression it is 10 wines better (176 against 166).

The cross-validation accuracy that TPOT measured on the training set was 0.637, and the test accuracy is 0.647. The two numbers are close, which is a good sign. It tells me the pipeline did not just memorise the training set and that the cross-validation score was an honest estimate of how it performs on new wines.

The pattern across the classes is the same as before. The Low class is the strongest, with a precision of 0.72 and a recall of 0.81. The Medium class has a precision of 0.57 and a recall of 0.54. The High class is still the weakest. Its recall is 0.38, so the pipeline found about 14 of the 37 High wines. This is a little better than my manual Random Forest (0.324), but far below the balanced Logistic Regression, which found 73% of them. TPOT was told to search for the best accuracy, so it did not pay special attention to the smallest class.

Overall, AutoML gave the best result, but the gain over my manual Random Forest is small, about 1.5 percentage points in accuracy. TPOT also chose a Random Forest as its final model, which confirms my manual choice. What it added was an automatic feature crossing step and tuned settings. This came at a cost. The search took several minutes, tested 116 pipelines and needed a separate Python environment, while my manual models trained in seconds. The winning pipeline also works with 77 features instead of 11, which makes it harder to explain.